<center> <h2> Using Kafkanator Fairness metrics table  </h2> </center>

<p> In this notebook you will learn to use kafkanator create_dataviz method to quickly get a summary of fairness metrics in your predictive model.
For this, we will use the ADULT dataset, a well known dataset in fairness domain to predict income based in some attributes, including sensistive ones such as gender and race. </p>
<h3> Outline : </h3>
<ol>
    <li> Data Downloading </li>
    <li> Data Preprocessing </li>
    <li> Training Model </li>
    <li> Executing the fairness metrics table method .</li>
    <ol>
        <li>A single sensitive attribute . </li>
        <li>Multiple sensitive attributes. </li>
</ol>

In [1]:
%pip install ucimlrepo


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<h3> 1. Data Downloading </h3>

In [2]:
from ucimlrepo import fetch_ucirepo 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from kafkanator.fairness.metrics import fairness_metrics_table
from collections import Counter
import xgboost as xgb
import pandas as pd
import numpy as np
from kafkanator.fairness import *
import kafkanator.util as ut

In [ ]:
# fetch dataset 
adult = fetch_ucirepo(id=2) 
# data (as pandas dataframes) 
data = adult.data.features 
y = adult.data.targets 
# metadata 
# variable information 
# Data cleaning
y['income'] = y['income'].apply(lambda x: x.replace('.',''))
data['sex'] = data['sex'].str.strip()
data['race'] = data['race'].str.strip()
data['marital-status'] = data['marital-status'].str.strip()
data['native-country'] = data['native-country'].str.strip()

In [4]:
data

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48837,39,Private,215419,Bachelors,13,Divorced,Prof-specialty,Not-in-family,White,Female,0,0,36,United-States
48838,64,NaN,321403,HS-grad,9,Widowed,NaN,Other-relative,Black,Male,0,0,40,United-States
48839,38,Private,374983,Bachelors,13,Married-civ-spouse,Prof-specialty,Husband,White,Male,0,0,50,United-States
48840,44,Private,83891,Bachelors,13,Divorced,Adm-clerical,Own-child,Asian-Pac-Islander,Male,5455,0,40,United-States


<h3> 2. Data preprocessing </h3>

In [ ]:
data['race'] = data['race'].apply(lambda x : 1 if x=='White' else 0 )

In [6]:
data[data['sex']=='Male'].shape

(32650, 14)

In [7]:
data[data['sex']=='Female'].shape

(16192, 14)

In [8]:
categorical_columns = ['workclass', 'education',  'occupation', 'relationship', 'race', 'sex','marital-status','native-country']

# Create an instance of LabelEncoder
label_encoder = LabelEncoder()

# Encode each categorical column and create new feature columns
for col in categorical_columns:
    encoded_col_name = f"{col}_encoded"
    data[encoded_col_name] = label_encoder.fit_transform(data[col])

In [9]:
Y = label_encoder.fit_transform(y['income'])

In [10]:
X = data[['age','workclass_encoded', 'education_encoded','education-num','marital-status_encoded','occupation_encoded',	'relationship_encoded','race_encoded','sex_encoded','capital-loss','capital-gain','hours-per-week','native-country_encoded']]

<h2> 3. Training the model </h2>

In [11]:
X_train, X_test, y_train, y_test = train_test_split( X, Y, test_size=0.3)

In [12]:
#classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier = xgb.XGBClassifier(objective="binary:logistic")
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

In [13]:
y_pred

array([0, 1, 1, ..., 0, 0, 0])

In [14]:
sens_metrics = X_test['sex_encoded']

In [15]:
X_test

,age,workclass_encoded,education_encoded,education-num,marital-status_encoded,occupation_encoded,relationship_encoded,race_encoded,sex_encoded,capital-loss,capital-gain,hours-per-week,native-country_encoded
29439,32,4,11,9,0,1,3,1,0,0,0,40,39
17387,29,4,9,13,4,10,1,1,1,0,8614,45,39
10550,46,4,12,14,2,4,0,1,1,0,0,45,39
34013,43,4,11,9,2,14,0,1,1,0,0,30,39
2980,37,4,12,14,4,4,1,1,1,0,0,40,39
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5286,25,4,6,5,5,6,2,1,1,0,0,40,8
25529,45,4,15,10,5,3,1,1,1,0,0,41,39
21187,31,4,15,10,2,8,0,1,1,0,0,40,39
28754,17,4,1,7,4,6,3,1,1,0,0,16,39


<h2> 4. Executing the fairness metrics table method </h2>
<p> This method receives as parameter a numpy matrix with sensitive attribute, prediction and reality columns. You can specify
one sensitive attribute in an array, or multiple. As indicated in followint subsections </p>

In [16]:
fairness_analysis = np.transpose(np.array([X_test['sex_encoded'].values, y_pred,y_test]))
fairness_analysis_df = pd.DataFrame(data=fairness_analysis,columns=['sex_encoded','prediction','reality'])

<h2> 4.1. One sensitive attribute : Gender </h2>

In [17]:
sp = fairness_metrics_table(fairness_analysis_df,['sex_encoded'],'prediction','reality',aggregate_metrics=False)

In [18]:
sp

,0,1
DEMOGRAPHIC PARITY - P1,0.088,0.258104
EQUAL OPPORTUNITY - TPR,0.599265,0.669571
PREDICTIVE PARITY - PPV,0.779904,0.78169
DISPARATE IMPACT - PREVALENCE,0.114526,0.301323


In [19]:
sp.style.apply(ut.default_row_highlighting,axis=1)

d is  0.17010360496819146
 in if
d is  0.07030633969405453
 in if
d is  0.0017858346249747248
 in if
d is  0.18679651567573885
 in if


,0,1
DEMOGRAPHIC PARITY - P1,0.088000,0.258104
EQUAL OPPORTUNITY - TPR,0.599265,0.669571
PREDICTIVE PARITY - PPV,0.779904,0.781690
DISPARATE IMPACT - PREVALENCE,0.114526,0.301323


<h2> You can also obtain normal fairness metrics by using kafkanator fairness module methods: </h2>

<h3> 4.2 Multiple sensitive attributes : gender and race </h3>
<p> Just build a numpy matrix containing sensitive attributes, reality and predictions columns, then pass an array containning
an arbitrary set of sensitive attributes </p>

In [21]:
fairness_analysis = np.transpose(np.array([X_test['sex_encoded'].values,X_test['race_encoded'].values, y_pred,y_test]))
concatenated = pd.DataFrame(data=fairness_analysis,columns=['sex_encoded','race_encoded','prediction','reality'])
sp = fairness_metrics_table(concatenated,['sex_encoded','race_encoded'],'prediction','reality',aggregate_metrics=False)

In [22]:
sp

,"0,0","0,1","1,0","1,1"
DEMOGRAPHIC PARITY - P1,0.050605,0.09685,0.187344,0.267869
EQUAL OPPORTUNITY - TPR,0.492063,0.613306,0.67037,0.669492
PREDICTIVE PARITY - PPV,0.673913,0.793011,0.804444,0.779494
DISPARATE IMPACT - PREVALENCE,0.069307,0.125228,0.224813,0.311882
